# Corvia Pipeline Intelligence Engine — Exploratory Analysis
**Company:** Corvia Technologies Ltd. | B2B SaaS, Compliance & Contract Intelligence  
**Period:** January 2023 – January 2025 (24-month history + live Q1 2025 pipeline)  
**Author:** Revenue Operations Analyst  

---
## Business Context

Corvia's VP of Sales submits a quarterly forecast to the board. In the last 8 quarters, that forecast has missed actuals by an average of **21.7%** — no quarter was explained, none was learned from.

This notebook explores 2,382 synthetic CRM opportunities to:
1. Quantify the rep forecast error vs a stage-velocity model alternative
2. Identify where deals stall (stage velocity analysis)
3. Understand win rate variation by segment and territory
4. Surface the at-risk deals in the current Q1 2025 pipeline

All data is synthetic, generated to mirror a real Salesforce CRM environment.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting config
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Load all data
opps  = pd.read_csv('../data/sample/fact_opportunities.csv', parse_dates=['create_date','estimated_close_date','actual_close_date'])
hist  = pd.read_csv('../data/sample/fact_stage_history.csv')
fcst  = pd.read_csv('../data/sample/fact_forecast_submissions.csv')
bench = pd.read_csv('../data/sample/stage_benchmarks.csv')
ar    = pd.read_csv('../data/sample/at_risk_deals.csv')
pipe  = pd.read_csv('../data/sample/q1_2025_pipeline_snapshot.csv')
users = pd.read_csv('../data/sample/dim_users.csv')

print(f"Opportunities: {len(opps):,} total")
print(f"  Closed Won:  {(opps['is_won']==True).sum():,}")
print(f"  Closed Lost: {(opps['is_won']==False).sum():,}")
print(f"  Open:        {opps['is_open'].sum():,} (Q1 2025 pipeline)")
print(f"Stage history: {len(hist):,} records")
print(f"Historical quarters: {len(fcst)}")


---
## 1. The Core Problem: Forecast Accuracy

The board asks: *"How much will we close this quarter?"*

The current answer is a survey of rep opinions — each AE classifies their deals as Commit, Best Case, or Pipeline. The VP totals the Commit column and calls it a forecast.

The following chart shows what that method produces vs what a stage-velocity model produces.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(fcst))
w = 0.35

bars1 = ax.bar(x - w/2, fcst['rep_forecast_variance_pct'], w,
               label='Rep-submitted forecast error %', color='#ef4444', alpha=0.82, zorder=3)
bars2 = ax.bar(x + w/2, fcst['model_forecast_variance_pct'], w,
               label='Stage-velocity model error %', color='#10b981', alpha=0.82, zorder=3)

ax.axhline(0, color='#334155', lw=0.8)
ax.axhline(-15, color='#ef4444', lw=0.8, ls='--', alpha=0.4)
ax.text(7.6, -15.8, '−15% industry threshold', fontsize=8, color='#ef4444', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(fcst['quarter'], fontsize=9)
ax.set_ylabel('Forecast variance vs actuals (%)', fontsize=10)
ax.set_title('Rep Forecast vs Stage-Velocity Model — 8 Quarters', fontsize=12, fontweight='bold', pad=12)
ax.legend(fontsize=9, framealpha=0.6)
ax.set_ylim(-30, 12)

# Annotate averages
avg_rep   = fcst['rep_forecast_variance_pct'].abs().mean()
avg_model = fcst['model_forecast_variance_pct'].abs().mean()
fig.text(0.13, 0.01, f'Rep avg error: {avg_rep:.1f}%   |   Model avg error: {avg_model:.1f}%   |   Improvement: {avg_rep-avg_model:.1f} ppts',
         fontsize=9.5, color='#475569', fontweight='bold')

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

print(f"\nRep average absolute error:   {avg_rep:.1f}%")
print(f"Model average absolute error: {avg_model:.1f}%")
print(f"Rep error was ALWAYS negative — consistently over-forecasting.")
print(f"Model stayed within ±7% every quarter.")


**Findings:**
- Rep-submitted forecast overestimates actuals by **20–24%** every quarter without exception — this is not noise, it's a systematic pattern (deal bloat and sandbagging cancel out differently each quarter, but the net result is always an over-forecast)
- The stage-velocity model achieves **3.8% average error** on the same 8 quarters using only historical win rates applied to pipeline stage
- The improvement is **17.9 percentage points** — significant enough to change how the board trusts the number

---
## 2. Win Rate Analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Win rate by segment
closed = opps[(opps['is_open']==False) & opps['is_won'].notna()].copy()
seg_wr = (closed.groupby('segment')
          .apply(lambda g: (g['is_won'].astype(float).sum()/len(g))*100)
          .reset_index(name='win_rate'))
seg_colors = {'SMB':'#3b82f6','Mid-Market':'#10b981','Enterprise':'#ef4444'}
axes[0].bar(seg_wr['segment'], seg_wr['win_rate'],
            color=[seg_colors.get(s,'grey') for s in seg_wr['segment']],
            alpha=0.85, zorder=3)
axes[0].axhline(28, color='#475569', lw=1, ls='--', label='Company avg 28%')
axes[0].set_title('Win Rate by Segment', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Win rate %', fontsize=10)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 50)
for i, row in seg_wr.iterrows():
    axes[0].text(i, row['win_rate']+0.8, f"{row['win_rate']:.1f}%",
                ha='center', fontsize=10, fontweight='bold')

# Right: Win rate by territory
terr_wr = (closed.groupby('territory')
           .apply(lambda g: (g['is_won'].astype(float).sum()/len(g))*100)
           .reset_index(name='win_rate')
           .sort_values('win_rate', ascending=True))
t_colors = ['#ef4444' if r < 28 else '#3b82f6' if r < 33 else '#10b981'
            for r in terr_wr['win_rate']]
axes[1].barh(terr_wr['territory'], terr_wr['win_rate'],
             color=t_colors, alpha=0.85, zorder=3)
axes[1].axvline(28, color='#475569', lw=1, ls='--', label='Company avg 28%')
axes[1].set_title('Win Rate by Territory', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Win rate %', fontsize=10)
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 45)
for i, row in terr_wr.iterrows():
    axes[1].text(row['win_rate']+0.4, list(terr_wr['territory']).index(row['territory']),
                f"{row['win_rate']:.1f}%", va='center', fontsize=9.5, fontweight='bold')

plt.suptitle('Win Rate Analysis — Corvia Technologies', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


**Findings:**
- Enterprise win rate (17.7%) is less than half of SMB (35.4%) — expected given longer cycles and stronger competition (Ironclad, Icertis)
- **Nordics underperforms at 25.8%** — territory is understaffed; one AE resigned 6 months ago and was not replaced
- **Benelux leads at 35.6%** — suggests the territory is undertaxed (quota too low relative to win rate)
- DACH at 29.2% is below UK/Ireland (32.3%) despite having 40% higher average ACV potential — a staffing and quota calibration problem

---
## 3. Stage Velocity — Where Deals Stall


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

stage_order = ['Prospect','Qualification','Discovery','Demo/Evaluation','Proposal','Negotiation']

# Left: Avg days heatmap
pivot_days = bench.pivot(index='stage_name', columns='segment', values='avg_days_in_stage')
pivot_days = pivot_days.reindex(stage_order)
sns.heatmap(pivot_days, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, linecolor='white', ax=axes[0],
            cbar_kws={'label':'Avg days'})
axes[0].set_title('Average Days per Stage', fontsize=11, fontweight='bold')
axes[0].set_xlabel(''); axes[0].set_ylabel('')
axes[0].set_xticklabels(axes[0].get_xticklabels(), fontsize=9)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, fontsize=9)

# Right: Stall rate heatmap
pivot_stall = bench.pivot(index='stage_name', columns='segment', values='stall_rate_pct')
pivot_stall = pivot_stall.reindex(stage_order)
sns.heatmap(pivot_stall, annot=True, fmt='.1f', cmap='Reds',
            linewidths=0.5, linecolor='white', ax=axes[1],
            cbar_kws={'label':'Stall rate %'})
axes[1].set_title('Stall Rate % per Stage', fontsize=11, fontweight='bold')
axes[1].set_xlabel(''); axes[1].set_ylabel('')
axes[1].set_xticklabels(axes[1].get_xticklabels(), fontsize=9)
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0, fontsize=9)

plt.suptitle('Stage Velocity — Avg Duration and Stall Rate by Segment', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Bottleneck identification
bench['bottleneck_score'] = bench['avg_days_in_stage'] * bench['stall_rate_pct'] / 100
top3 = bench.nlargest(3, 'bottleneck_score')[['segment','stage_name','avg_days_in_stage','stall_rate_pct','bottleneck_score']]
print("\nTop 3 pipeline bottlenecks (avg_days × stall_rate):")
print(top3.to_string(index=False))


**Findings:**
- **Mid-Market Demo/Evaluation is the primary bottleneck**: 16.3 avg days, 13.2% stall rate — the highest stall rate in the Mid-Market funnel
- Stalled MM Demo deals spend an average of **27.5 days** vs 16.3 for normal deals — a 68% longer cycle
- **Enterprise Negotiation has the highest overall bottleneck score**: 29.1 avg days, 13.7% stall — this is the deal desk problem (long internal approval cycles, Project 5)
- SMB funnel is relatively healthy — lower durations and stall rates across all stages

---
## 4. Current Q1 2025 Pipeline Snapshot


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Pipeline by stage (face value vs model)
stage_grp = (pipe.groupby(['current_stage','current_stage_name'])
             .agg(deals=('opportunity_id','count'),
                  face_value=('acv_eur','sum'),
                  model=('model_weighted','sum'))
             .reset_index().sort_values('current_stage'))

x = np.arange(len(stage_grp))
w = 0.38
axes[0].bar(x-w/2, stage_grp['face_value']/1e6, w,
            label='Face-value ACV (€M)', color='#93c5fd', alpha=0.9, zorder=3)
axes[0].bar(x+w/2, stage_grp['model']/1e6, w,
            label='Model forecast (€M)', color='#10b981', alpha=0.85, zorder=3)
for i, (_, row) in enumerate(stage_grp.iterrows()):
    axes[0].text(i, row['face_value']/1e6+0.02, f"{int(row['deals'])}",
                ha='center', fontsize=8, color='#475569')
axes[0].set_xticks(x)
axes[0].set_xticklabels(stage_grp['current_stage_name'], rotation=20, ha='right', fontsize=8.5)
axes[0].set_ylabel('Value (€M)', fontsize=10)
axes[0].set_title('Q1 2025 Pipeline — 167 Open Deals', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].text(0, -0.38, 'Numbers above bars = deal count', fontsize=7.5,
             color='#64748b', transform=axes[0].transData)

# Right: At-risk breakdown
ar_open = ar[ar['is_open']==True] if 'is_open' in ar.columns else ar
risk_counts = ar.groupby('risk_tier')['acv_eur'].agg(['count','sum']).reset_index()
risk_counts.columns = ['tier','count','acv']
colors_r = {'HIGH':'#ef4444','MEDIUM':'#f59e0b','NORMAL':'#10b981'}
for i, row in risk_counts.iterrows():
    c = colors_r.get(row['tier'],'grey')
    axes[1].bar(row['tier'], row['count'], color=c, alpha=0.8, zorder=3)
    axes[1].text(i, row['count']+0.3, f"{int(row['count'])}\n€{row['acv']/1e3:.0f}K",
                ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_ylabel('Deal count', fontsize=10)
axes[1].set_title('At-Risk Deal Distribution', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Risk tier', fontsize=10)

plt.suptitle('Q1 2025 — Pipeline Snapshot & Risk Profile', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

total_pipe = pipe['acv_eur'].sum()
model_fcst = pipe['model_weighted'].sum()
quota      = users['quota_arr_quarterly'].sum()
print(f"Total pipeline (face value): €{total_pipe:,.0f}")
print(f"Model forecast (weighted):   €{model_fcst:,.0f}")
print(f"Quarterly quota:             €{quota:,.0f}")
print(f"Coverage ratio:              {total_pipe/quota:.2f}× (need 3.57×)")
print(f"Rep commit (CRM):            €{pipe[pipe['forecast_category']=='Commit']['acv_eur'].sum():,.0f}")


**Findings:**
- **Pipeline coverage at 1.77×** — significantly below the required 3.57× at our 28% win rate
- Stage 4 (Demo/Evaluation) holds the most face-value ACV (€2.5M) but only contributes €949K to the model forecast (38% win probability)
- Rep-submitted commit (€879K) is €1.97M below the model forecast — **sandbagging signal**: reps are classifying likely-to-close deals as Best Case
- **14 deals (€1.02M ACV) are stalled** beyond their stage threshold; 9 are HIGH risk

---
## 5. Key Findings Summary


In [ ]:
print("="*60)
print("CORVIA FORECASTING INTELLIGENCE — KEY FINDINGS")
print("="*60)
print()
print("1. FORECAST ACCURACY")
print("   Rep forecast error:   21.7% avg (8 quarters)")
print("   Model forecast error:  3.8% avg (same period)")
print("   Improvement:          17.9 percentage points")
print()
print("2. PIPELINE COVERAGE (Q1 2025)")
print("   Current coverage:     1.77× (need 3.57×)")
print("   Gap:                 -1.80× — structurally at risk")
print()
print("3. STAGE BOTTLENECK")
print("   MM Demo/Evaluation:   16.3 avg days, 13.2% stall rate")
print("   Stalled deals:        27.5 avg days (68% longer)")
print()
print("4. TERRITORY GAPS")
print("   Nordics win rate:     25.8% — understaffed territory")
print("   DACH win rate:        29.2% vs UK/Ireland 32.3%")
print("   DACH has 40% higher ACV potential but 2 fewer AEs")
print()
print("5. AT-RISK PIPELINE")
print("   14 stalled deals, €1.02M ACV at risk of Q2 slip")
print("   9 HIGH risk (P90+ stage duration)")
print()
print("RECOMMENDATION: Adopt model-generated forecast as board")
print("submission baseline. Rep commit remains visible but is")
print("audited against model on all HIGH-variance quarters.")
